# 2.9 ONNX Embedder: Same Pipeline, No PyTorch

`sentence-transformers` drags in PyTorch and a pile of NVIDIA libraries:

| Setup                 | Size   | Packages |
| --------------------- | ------ | -------- |
| sentence-transformers | 4.8 GB | 58       |
| ONNX Runtime          | 147 MB | 27       |

ONNX Runtime is **33x smaller** and produces identical embeddings.
For experiments, sentence-transformers is fine; for production, use the lightweight option.

The model (`Xenova/all-MiniLM-L6-v2`) is already stored locally under `models/Xenova/all-MiniLM-L6-v2/`.
If not: `uv run python download.py`

## The Embedder

`embedder.py` wraps four steps:

1. **Tokenize** - convert text into integer IDs and attention masks
2. **ONNX inference** - execute the model graph on CPU
3. **Mean pooling** - average token embeddings weighted by the attention mask
4. **L2 normalize** - divide by L2 norm so dot product equals cosine similarity

The interface (`encode`, `encode_batch`) is identical to `SentenceTransformer.encode()`.

In [7]:
import numpy as np
from numpy.typing import NDArray
from embedder import Embedder

Embedding = NDArray[np.float32]

embed = Embedder()

## Comparing two queries against a document

In [ ]:
q1: str  = "Can I still join the course after the start date?"
q2: str  = "How to install Docker on Windows?"
d: str    = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1: Embedding = embed.encode(q1)
v2: Embedding = embed.encode(q2)
dv: Embedding = embed.encode(d)

(384,) (384,) (384,)
float64 float64 float64


In [11]:
# q1 vs document — high similarity (same topic)
sim_1d = v1.dot(dv)
print(f"q1 ~ d: {sim_1d:.4f}")

q1 ~ d: 0.3233


In [12]:
# q2 vs document — low similarity (different topic)
sim_2d = v2.dot(dv)
print(f"q2 ~ d: {sim_2d:.4f}")

q2 ~ d: 0.0197


Same values as with sentence-transformers. `q1` scores higher because the course-joining query is semantically closer to the registration document.

## Embedding the FAQ dataset

In [ ]:
import sys
sys.path.insert(0, "..")

from ingest import load_faq_data

documents = load_faq_data()
print(f"{len(documents)} documents loaded")

In [ ]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [ ]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)
X.shape

## Search

In [ ]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

In [ ]:
# Top-5 results
top5 = np.argsort(-scores)[:5]
for i in top5:
    print(f"[{scores[i]:.4f}] {documents[i]['question']}")

Same results, same pipeline, ~33x lighter.

## Available models

All work with the same code — just change the model name in `download.py` and the path in `Embedder()`:

| Model | Dim | Notes |
|---|---|---|
| `Xenova/all-MiniLM-L6-v2` | 384 | Best small general-purpose |
| `Xenova/all-MiniLM-L12-v2` | 384 | Better quality, slower |
| `Xenova/bge-small-en-v1.5` | 384 | Strong retrieval |
| `Xenova/bge-base-en-v1.5` | 768 | Stronger retrieval |
| `Xenova/multilingual-e5-small` | 384 | Multilingual retrieval |
| `Xenova/gte-small` | 384 | Lightweight modern model |

In [ ]:
# Example: load a different model
# embed_bge = Embedder("models/Xenova/bge-base-en-v1.5")
# embed_bge.encode("your text here").shape  # (768,)